In [2]:
!pip install langchain langchain-core langchain-google-genai langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('customer_reviews_dataset_45.csv')
df.head()

In [ ]:
df.rename(columns={'Rating (1-5)':'Rating','Review Title':'Review'},inplace=True)

In [ ]:
df

,Cust_Id,Product,Rating,Review,Full Review Text,Rating_Type
0,CUST1001,Phone Case,4,Works as expected,Easy to use and performs exactly as described....,Average or Good
1,CUST1002,Laptop Stand,2,Cheap quality,The material feels flimsy and not worth the pr...,Critical
2,CUST1003,Phone Case,3,Damaged package,The box was damaged when it arrived although t...,Average or Good
3,CUST1004,Phone Case,5,Amazing product,Exceeded my expectations in every way. I have ...,Average or Good
4,CUST1005,Wireless Mouse,2,Cheap quality,The material feels flimsy and not worth the pr...,Critical
5,CUST1006,Water Bottle,5,Excellent value,Great quality for the price. Highly recommende...,Average or Good
6,CUST1007,Backpack,5,Easy to install,Setup took only a few minutes and everything w...,Average or Good
7,CUST1008,Bluetooth Speaker,2,Cheap quality,The material feels flimsy and not worth the pr...,Critical
8,CUST1009,Laptop Stand,2,Poor battery,Battery life is much shorter than advertised. ...,Critical
9,CUST1010,Phone Case,3,Damaged package,The box was damaged when it arrived although t...,Average or Good


In [ ]:
df['Rating'].value_counts()

Rating
4    12
2    12
5    12
3     8
1     1
Name: count, dtype: int64

In [ ]:
df['Rating_Type'] = np.where(df['Rating'].isin([1,2]),'Critical','Average or Good')


In [ ]:
df['Rating_Type'].value_counts()

Rating_Type
Average or Good    32
Critical           13
Name: count, dtype: int64

In [ ]:
critical_df = df[df['Rating_Type']=='Critical']
critical_df.shape

(13, 6)

In [ ]:
from collections import Counter

In [ ]:
df.columns

Index(['Cust_Id', 'Product', 'Rating', 'Review', 'Full Review Text',
       'Rating_Type'],
      dtype='object')

In [ ]:
def most_common_complain_keywords(data):
  keyword_counter = Counter(data['Review'])
  return keyword_counter

In [ ]:
critical_keyword = most_common_complain_keywords(critical_df)
critical_keyword

Counter({'Cheap quality': 7, 'Poor battery': 5, 'Wrong item': 1})

In [ ]:
df[df['Review']=='Poor Battery']

,Cust_Id,Product,Rating,Review,Full Review Text,Rating_Type


In [ ]:
ck_df = pd.DataFrame(critical_keyword.items(),columns=['Keywords','Count'])
ck_df

,Keywords,Count
0,Cheap quality,7
1,Poor battery,5
2,Wrong item,1


In [ ]:
import plotly.express as px

In [ ]:
fig = px.bar(ck_df,x='Keywords',y='Count',text='Count')
fig.show()

In [ ]:
def topk(data,K):
  top3_crit_reviews = data.sort_values(by='Count',ascending=False).head(K)
  return top3_crit_reviews

In [ ]:
topk(ck_df,3)

,Keywords,Count
0,Cheap quality,7
1,Poor battery,5
2,Wrong item,1


Top 3 most common critical keywords

1) Cheap Quality
2) Poor Battery
3) Wrong item

In [ ]:
df.columns

Index(['Cust_Id', 'Product', 'Rating', 'Review', 'Full Review Text',
       'Rating_Type'],
      dtype='object')

In [ ]:
def extract_cust_id_and_review(data,Cust_ID):
  text = data['Review'][data['Cust_Id']==Cust_ID]
  return text


In [ ]:
tv=extract_cust_id_and_review(df,'CUST1002')
tv

1    Cheap quality
Name: Review, dtype: object

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

In [1]:
import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

In [ ]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash',temperature=0.72,
                             api_key=os.environ['GEMINI_API_KEY'])

parser = StrOutputParser()

In [ ]:
prompt  = PromptTemplate.from_template("""
Role
You are an experienced Customer Support Agent for an e-commerce company.

Task
Generate a short, personalized, professional, and empathetic apology email for the customer based on the provided Customer ID and Review.

Inputs
Customer Review: {Review}
Instructions
Address the customer using their Customer ID.
Begin with a sincere apology.
Specifically acknowledge the complaint mentioned in the review. Do not generate a generic apology.
Show empathy and understanding.
Briefly assure the customer that the issue will be investigated and steps will be taken to prevent it from happening again.
If applicable, mention that the customer can contact the support team for additional assistance.
End the email with a polite closing and appreciation for the customer's feedback.
Keep the email between 80 and 120 words.
Maintain a warm, respectful, and professional tone.
Do not invent order details, dates, refund amounts, or promises that are not supported by the review.
Output Format

Dear {Cust_ID}

We're Sorry About Your Experience


<Generate a personalized apology email here based on the review.>

Best Regards,

Customer Support Team

""")

In [ ]:
chain = prompt | llm | parser

text_review = extract_cust_id_and_review(df,'CUST1043')
res = chain.invoke({'Cust_ID':'CUST1043','Review':text_review})
print(res)

Dear CUST1043,

We're Sorry About Your Experience

We are truly sorry to hear about the issue you experienced with your recent order. We understand how frustrating it must be to receive the wrong item, and this is certainly not the experience we want for our valued customers.

Please be assured that we are taking your feedback seriously and will investigate what went wrong to prevent similar occurrences in the future. Our team is ready to assist you further with this specific issue. Please reach out to our support team at your earliest convenience so we can work towards a resolution for you.

Thank you for bringing this to our attention. Your feedback is invaluable in helping us improve our service.

Best Regards,

Customer Support Team


3 AI generated mails:-

1) Cheap Quality<br>
Dear CUST1002,

We're Sorry About Your Experience

Please accept our sincerest apologies regarding your recent purchase. We are truly sorry to hear that the product you received was of "cheap quality," and we understand how frustrating and disappointing this must be. Your satisfaction is incredibly important to us, and it's clear we fell short in this instance.

We are taking your feedback seriously and will be thoroughly investigating this issue to understand what went wrong and to prevent similar occurrences in the future. We are committed to providing high-quality products, and your comments are invaluable in helping us improve.

Should you wish to discuss this further or require any additional assistance, please do not hesitate to contact our support team.

Thank you for bringing this to our attention.

Best Regards,

Customer Support Team




2) Poor Battery<br>

Dear CUST1022   

We're Sorry About Your Experience

Please accept our sincerest apologies for the poor battery performance you highlighted in your recent review. We truly understand how frustrating such an issue can be when a product doesn't meet expectations. Your feedback is vital, and we are taking this seriously. We will investigate this matter thoroughly to understand the cause and implement measures to prevent it from happening again. If you'd like to discuss this further or need additional assistance, please feel free to contact our support team directly. Thank you for your valuable feedback; it helps us continuously improve.

Best Regards,

Customer Support Team

3) Wrong Item<br>
Dear CUST1043,

We're Sorry About Your Experience

We are truly sorry to hear about the issue you experienced with your recent order. We understand how frustrating it must be to receive the wrong item, and this is certainly not the experience we want for our valued customers.

Please be assured that we are taking your feedback seriously and will investigate what went wrong to prevent similar occurrences in the future. Our team is ready to assist you further with this specific issue. Please reach out to our support team at your earliest convenience so we can work towards a resolution for you.

Thank you for bringing this to our attention. Your feedback is invaluable in helping us improve our service.

Best Regards,

Customer Support Team